在 PyTorch 中，optimizer.step() 是优化器（Optimizer）的一个关键方法，用于更新模型的参数。
- 它根据计算得到的梯度（存储在每个参数的 .grad 属性中）来调整模型的参数，从而最小化损失函数。
- 这个方法是训练神经网络过程中不可或缺的一部分。

optimizer.step() 的主要作用是：
- 更新参数：根据计算得到的梯度，更新模型的参数。
- 支持多种优化算法：PyTorch 提供了多种优化器（如 SGD、Adam、RMSprop 等），每种优化器都有自己的更新规则。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 定义一个简单的线性模型
class LinearModel(nn.Module):
    def __init__(self):
        super(LinearModel, self).__init__()
        self.linear = nn.Linear(1, 1)  # 输入维度为1，输出维度为1

    def forward(self, x):
        return self.linear(x)

# 创建模型实例
model = LinearModel()

# 定义损失函数和优化器
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 假设有一些输入数据和目标数据
inputs = torch.tensor([[1.0], [2.0], [3.0]], requires_grad=True)
targets = torch.tensor([[2.0], [4.0], [6.0]])

### 前向传播和反向传播

In [ ]:
# 前向传播
outputs = model(inputs)
loss = criterion(outputs, targets)

# 反向传播
optimizer.zero_grad()  # 清空之前的梯度
loss.backward()        # 计算梯度

# 更新参数
optimizer.step()       # 更新模型参数

### 优化器的选择：
PyTorch 提供了多种优化器，每种优化器都有自己的更新规则。常见的优化器包括：
- SGD：随机梯度下降
- Adam：自适应矩估计
- RMSprop：均方根传播
- Adagrad：自适应梯度算法
- LBFGS：有限内存 BFGS 算法

### 使用 closure 的例子
- 某些优化器（如 LBFGS）需要多次计算损失来调整参数。
- 在这种情况下，可以使用 closure 参数。
- closure 是一个重新计算模型损失的函数，通常用于需要多次计算损失的优化器。

In [ ]:
# 定义一个简单的线性模型
class LinearModel(nn.Module):
    def __init__(self):
        super(LinearModel, self).__init__()
        self.linear = nn.Linear(1, 1)  # 输入维度为1，输出维度为1

    def forward(self, x):
        return self.linear(x)

# 创建模型实例
model = LinearModel()

# 定义损失函数和优化器
criterion = nn.MSELoss()
optimizer = optim.LBFGS(model.parameters(), lr=0.01)

# 假设有一些输入数据和目标数据
inputs = torch.tensor([[1.0], [2.0], [3.0]], requires_grad=True)
targets = torch.tensor([[2.0], [4.0], [6.0]])

# 定义 closure 函数
def closure():
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    return loss

# 更新参数
optimizer.step(closure)

In [ ]:
# 创建一个很简单的网络：两个卷积层，一个全连接层
model = Simple()
# 为了方便观察数据变化，把所有网络参数都初始化为 0.1
for m in model.parameters():
    m.data.fill_(0.1)
 
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1.0)
 
model.train()
# 模拟输入8个 sample，每个的大小是 10x10，
# 值都初始化为1，让每次输出结果都固定，方便观察
images = torch.ones(8, 3, 10, 10)
targets = torch.ones(8, dtype=torch.long)
output = model(images)
print(output.shape)
# torch.Size([8, 20])
 
loss = criterion(output, targets)
 
print(model.conv1.weight.grad)
# None
loss.backward()###############################################################
print(model.conv1.weight.grad[0][0][0])
# tensor([-0.0782, -0.0842, -0.0782])
# 通过一次反向传播，计算出网络参数的导数，
# 因为篇幅原因，我们只观察一小部分结果
 
print(model.conv1.weight[0][0][0])
# tensor([0.1000, 0.1000, 0.1000], grad_fn=<SelectBackward>)
# 我们知道网络参数的值一开始都初始化为 0.1 的
 
optimizer.step()###########################################################
print(model.conv1.weight[0][0][0])
# tensor([0.1782, 0.1842, 0.1782], grad_fn=<SelectBackward>)
# 回想刚才我们设置 learning rate 为 1，这样，
# 更新后的结果，正好是 (原始权重 - 求导结果 * 学习率 ) ！
 
optimizer.zero_grad()############每次更新完权重之后，我们记得要把导数清零啊，
# 不然下次会得到一个和上次计算一起累加的结果。
print(model.conv1.weight.grad[0][0][0])
# tensor([0., 0., 0.])
# 每次更新完权重之后，我们记得要把导数清零啊，
# 不然下次会得到一个和上次计算一起累加的结果。
# 当然，zero_grad() 的位置，可以放到前边去，
# 只要保证在计算导数前，参数的导数是清零的就好。